In [1]:
# Requirements
!pip install spotipy selenium pandas adjustText networkx matplotlib

In [1]:
# Import required libraries
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials, SpotifyOAuth
from spotipy.exceptions import SpotifyException
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import random
import re
import pprint
import json
import numpy as np
from adjustText import adjust_text
import os
import time
from datetime import datetime
from google.colab import files

# **Spotify Authorization Workflow: Credential Setup**

In [2]:
# Define Spotify API credentials
CLIENT_ID = '829be16fb27f43fd98c2332fcd335f7e'
CLIENT_SECRET = '309db9acfc034e08bb93d646aa13a120'
REDIRECT_URI = 'https://google.com/'

# Define scope
scope = 'playlist-read-private playlist-read-collaborative playlist-modify-private playlist-modify-public'

# Create OAuth object without automatic opening
sp_oauth = SpotifyOAuth(client_id=CLIENT_ID,
                        client_secret=CLIENT_SECRET,
                        redirect_uri=REDIRECT_URI,
                        scope=scope,
                        show_dialog=True)

# Generate the authorization URL
auth_url = sp_oauth.get_authorize_url()

# Manually display the authorization URL
print(f"Open this URL to authorize: {auth_url}")

# Paste the URL you get after authorization
redirected_url = input("Paste the redirected URL here: ")

# Extract authorization code from URL
code = sp_oauth.parse_response_code(redirected_url)

# Get token from the code using get_cached_token()
token_info = sp_oauth.get_cached_token()

# If no token is cached, fetch a new one
if not token_info:
    token_info = sp_oauth.get_access_token(code)

# Set the authorized Spotify object
sp = spotipy.Spotify(auth=token_info['access_token'])

print("✅ Authorization successful!")

Open this URL to authorize: https://accounts.spotify.com/authorize?client_id=829be16fb27f43fd98c2332fcd335f7e&response_type=code&redirect_uri=https%3A%2F%2Fgoogle.com%2F&scope=playlist-read-private+playlist-read-collaborative+playlist-modify-private+playlist-modify-public&show_dialog=True
Paste the redirected URL here: https://www.google.com/?code=AQAznecvEJO88NjmUC-we6URlaNga6Gv5ilKarhp29bDrd4YU5cff-McEhKWa_FjuO809XPMa3dyEEdXNh3ircbRue95ynIQO8yxECIJTgLAJCLVX5Cz2TRLZynQpKEZdMeXun4FgQ7gSg6fPta-_vbYFA2u4z8npP3KgNfTsKGTC4JA7HNFTFpU_8YZEUt1o68cxRIuzJflmXqtBtc7VhzYZiSVS-hsuyCqCkJ5iAwyHeY40-vDAepKeFVBNilHF2VmiCOeXfIeOPdQ1mZVumGgRrp1g7JIjhk4aw
✅ Authorization successful!


# **Caching**

In [3]:
CACHE_FILE = 'cache.json'

def load_cache():
  if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, 'r') as f:
            return json.load(f)
  return {}

def save_cache(cache_data):
    with open(CACHE_FILE, 'w') as f:
        json.dump(cache_data, f)

def check_cache(key, cache):
  if key in cache:
    return cache[key]
  return None

def update_cache(key, value, cache):
  cache[key] = value

# **Extracting Initial Artist Set**
This initial extraction only applies when using new playlists / creating the graph for the first time

In [4]:
# Get playlist tracks
def get_artists_from_playlist(playlist_id):
  # Check cache
  response = check_cache(playlist_id, cache)
  if response:
    print("Cache hit for playlist.")
  # If not in cache
  if response is None:
    response = sp.playlist_tracks(playlist_id)
    # while True:
    #     try:

    #         time.sleep(250)
    #         break
    #     except SpotifyException as e:
    #         if e.http_status == 429:
    #             retry_after = int(e.headers.get("Retry-After", 5))
    #             print(f"Rate limit hit. Retrying after {retry_after} seconds...")
    #             time.sleep(retry_after)
    #         else:
    #             raise e

    # Cache playlist
    update_cache(playlist_id, response, cache)
    save_cache(cache)
    # Cache individual tracks
    for item in response['items']:
      # Collect track info
      name = item['track']['name']
      album = item['track']['album']['name']
      release_date = item['track']['album']['release_date']
      track_artists = []
      for artists in item['track']['artists']:
        artist = artists['name']
        track_artists.append(artist)

      track_info = {
          'Artists': track_artists,
          'Album': album,
          'Release Date': str(release_date)
      }
      print("Caching track: ", name)
      update_cache(name, track_info, cache)
      save_cache(cache)

  tracks = response['items']

  artist_list = []

  for item in tracks:
      track = item['track']
      for artist in track['artists']:
          artist_list.append(artist['name'])

  return list(set(artist_list))  # Remove duplicates

In [6]:
  # Extract the playlist ID
# Spotify banned access to Featured playlists, so only using user-created playlists
def extract_playlist_id():
    url = input("Enter URL to your playlist (cannot be a Spotify Featured playlist)")
    match = re.search(r"playlist/([a-zA-Z0-9]+)", url)
    if match:
        return match.group(1)
    else:
        return None

In [7]:
cache = load_cache()

playlist_id = extract_playlist_id()
if playlist_id:
    print("Playlist ID:", playlist_id)
    artist_list = get_artists_from_playlist(playlist_id)
    print(f"Found {len(artist_list)} unique artists in the playlist.")
    print(artist_list)
else:
    print("No playlist ID found.")

save_cache(cache)

Enter URL to your playlist (cannot be a Spotify Featured playlist)https://open.spotify.com/playlist/1LDcLb8MX7Cq13DghRYNn9?si=029a9a4227094e5d
Playlist ID: 1LDcLb8MX7Cq13DghRYNn9
Caching track:  luther (with sza)
Caching track:  Dark Thoughts
Caching track:  NOKIA
Caching track:  RATHER LIE (with The Weeknd)
Caching track:  Anxiety
Caching track:  EVIL J0RDAN
Caching track:  tv off (feat. lefty gunplay)
Caching track:  Sports car
Caching track:  Die With A Smile
Caching track:  Ordinary
Caching track:  Just In Case
Caching track:  Timeless (feat Playboi Carti)
Caching track:  BIRDS OF A FEATHER
Caching track:  Not Like Us
Caching track:  Pink Pony Club
Caching track:  Sailor Song
Caching track:  All The Stars (with SZA)
Caching track:  That’s So True
Caching track:  No One Noticed
Caching track:  The Giver
Caching track:  Good Luck, Babe!
Caching track:  DENIAL IS A RIVER
Caching track:  DtMF
Caching track:  APT.
Caching track:  Beautiful Things
Caching track:  I'm The Problem
Caching 

# **Expanding the features and artist list**

In [8]:
# Extract features for given artist
# Current features: id, popularity, followers, genres
def get_artist_features(artist):
  # Check cache
  response = check_cache(artist, cache)
  if response:
    print("Cache hit for ", artist)
  # If not in cache
  if response is None:
    response = sp.search(q='artist:' + artist, type='artist')['artists']['items'][0]

    discography = get_artist_discography(response['id'])

    info = {
        'ID': response['id'],
        'Popularity': response['popularity'],
        'Followers': response['followers']['total'],
        'Genres': response['genres']

    }
    update_cache(artist, info, cache)
    save_cache(cache)
  else:
    # If artist is found in cache, assign response to info
    info = response
  return info

In [9]:
# Extract all features about all artists in the given list
def update_features(artist_list, artist_dictionary={}):
  for artist in artist_list:
    # Add artist and relevant features if not already in dictionary
    if artist not in artist_dictionary.keys():
      print("Adding ", artist)
      artist_dictionary[artist] = get_artist_features(artist)
  return artist_dictionary

In [13]:
# Helper function for getting an artist's full discography
def get_artist_discography(artist_id):
  discography = {}
  album_offset = 0
  limit = 50
  while True:
    album_results = sp.artist_albums(artist_id, album_type='album', limit=limit, offset=album_offset)
    time.sleep(250)
    # while True:
    #     try:
    #         # Get all albums by the artist

    #         time.sleep(250)
    #         break
    #     except SpotifyException as e:
    #         if e.http_status == 429:
    #             retry_after = int(e.headers.get("Retry-After", 5))
    #             print(f"Rate limit hit. Retrying after {retry_after} seconds...")
    #             time.sleep(retry_after)
    #         else:
    #             raise e

    # Get all songs for each album
    for album in album_results['items']:
      track_offset = 0
      while True:
        track_results = sp.album_tracks(album['id'], limit=limit, offset=track_offset)
        time.sleep(250)
        # try:

        #     time.sleep(250)
        #     break
        # except SpotifyException as e:
        #     if e.http_status == 429:
        #         retry_after = int(e.headers.get("Retry-After", 5))
        #         print(f"Rate limit hit. Retrying after {retry_after} seconds...")
        #         time.sleep(retry_after)
        #     else:
        #         raise e

        for track in track_results['items']:
          # Check if tracks are cached
          response = check_cache(track['name'], cache)
          if response:
            print("Cache hit for ", track['name'])
          if response is None:
            print("Adding track ", track['name'])
            name = track['name']
            artists = track['artists']
            album_name = album['name']
            release_date = album['release_date']

            track_info = {
                'Artists': artists,
                'Album': album_name,
                'Release Date': release_date
            }
            # Cache track
            update_cache(name, track_info, cache)
            save_cache(cache)
          else:
            track_info = response

          # Update discography
          discography[track['name']] = track_info

        # Paginate through tracks
        if track_results['next']:
          track_offset += limit
        else:
          break

    # Paginate through albums
    if album_results['next']:
      album_offset += limit
    else:
      break
  return discography

In [ ]:
# Test if sp.artist_albums works
artist_id = "your_artist_id"  # Replace with a real artist ID
try:
    albums = sp.artist_albums(artist_id, album_type='album', limit=5)
    print(albums)
except SpotifyException as e:
    print(f"Error: {e}")

In [11]:
# Find related artists (featured on current artists' songs)
def get_related_artists(artist_list, artist_dictionary):
  artist_relations = {}
  for artist in artist_dictionary:
    artist_id = artist_dictionary[artist]['ID']
    artist_discography = get_artist_discography(artist_id)

    # Collect artists based on discography
    related_artists = []
    for track in artist_discography:
      # Collect related artists for a given artist
      for artist_info in artist_discography[track]['Artists']:
        # Collect related artists for a given artist
        # Handle cases where 'Artists' is a list or a string
        if isinstance(artist_info, list):
          for r_artist in artist_info:
            if r_artist != artist:  # Check for dict and 'name' key
              related_artists.append(r_artist)
        elif artist_info != artist:  # Check for dict and 'name' key
          related_artists.append(artist_info)

        artist_relations[artist] = list(set(related_artists))

      # # Create relation between given artist and related artists
      # for artist_info in artist_discography[track]['Artists']:
      #   if artist_info['name'] in related_artists:
      #     artist_relations[artist] = artist_relations[artist] + related_artists
  return artist_relations

In [16]:
# Initializing artist graph and dictionary
artist_graph = nx.Graph()

# Maintaining network data for easy access
# Nodes is a list of dictionaries of structure {'name', 'weight'}
# Edges is a list of dictionaries of structure {'source', 'target'}

network_data = {
    'nodes': [],
    'edges': []
}

# with open("artist_network.json", "r") as f:
#     artist_dictionary = json.load(f)

artist_dictionary = update_features(artist_list) #, artist_dictionary)
artist_relations = get_related_artists(artist_list, artist_dictionary)
print(json.dumps(artist_relations, indent=4))
save_cache(cache)

Adding  ROSÉ


KeyboardInterrupt: 

In [ ]:
save_cache(cache)

In [ ]:
#color_map = []

# Add all the related artists into the artist list and update features
related_artists = []
for artist in artist_relations:
  for related_artist in artist_relations[artist]:
    if related_artist not in artist_list and related_artist not in related_artists:
      #print("Adding ", related_artist)
      related_artists.append(related_artist)

artist_list.extend(related_artists)
print("Newly added artists: ", related_artists)
print("Total artist list: ", artist_list)
artist_dictionary = update_features(artist_list, artist_dictionary)

# Assigning size of nodes by relative weights
weights = np.array([int(artist_dictionary[artist]['Followers']) for artist in artist_dictionary])
weights = weights / np.linalg.norm(weights)

# Create empty relations set for all artists
for artist in artist_dictionary:
  artist_dictionary[artist]['Related Artists'] = {}
  artist_dictionary[artist]['Related Artists']['Artist Collaboration'] = []

# Add all the relations into artist_dicitionary
for artist in artist_relations:
  if artist_relations[artist] != [] or artist_relations[artist] is not None:
    for related_artist in artist_relations[artist]:

      # Add to network data
      network_data['edges'].append({'source': artist, 'target': related_artist})

      # Add related artist to artist collabs
      if artist_dictionary[artist]['Related Artists']['Artist Collaboration'] == None:
        print("Adding ", related_artist)
        artist_dictionary[artist]['Related Artists']['Artist Collaboration'] = [related_artist]
      else:
        artist_dictionary[artist]['Related Artists']['Artist Collaboration'].append(related_artist)

      # Add artist to related artist collabs
      if artist_dictionary[related_artist]['Related Artists']['Artist Collaboration'] == None:
        artist_dictionary[related_artist]['Related Artists']['Artist Collaboration'] = [artist]
      else:
        artist_dictionary[related_artist]['Related Artists']['Artist Collaboration'].append(artist)

network_data['nodes'] = [{'name': name, 'weight': weight} for name, weight in zip(artist_list, weights)]

In [ ]:
# Create graph
for node in network_data['nodes']:
    artist_graph.add_node(node['name'], name=node['name'])
for edge in network_data['edges']:
    artist_graph.add_edge(edge['source'], edge['target'], weight=edge['weight'])

# Draw the graph
plot = plt.figure(figsize=(30,30))

# Use spring layout with adjusted k for more space
pos = nx.kamada_kawai_layout(artist_graph)

# Draw nodes
nx.draw(artist_graph, pos, with_labels=False, node_size=weights*10e3)

# Add labels
labels = nx.draw_networkx_labels(artist_graph, pos, font_size=6)#, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

# Adjust overlapping text
# texts = list(labels.values())
# adjust_text(texts)

plot.show()

In [ ]:
# print("Entire updated artist dictionary")
# print(json.dumps(artist_dictionary, indent=4))

In [ ]:
# Show a subgraph centered around the given artist
plt.figure(figsize=(10, 10))
artist = 'The Weeknd'
radius = 1
subgraph = nx.ego_graph(artist_graph, artist, radius=radius)

# Draw the subgraph
pos = nx.kamada_kawai_layout(subgraph)
nx.draw(subgraph, pos, with_labels=True, node_color="lightblue", node_size=2000, font_size=8)
plt.title(f"Subgraph around {artist} (radius={radius})")
plt.show()

# **Graph Expansion**

In [ ]:
# Recursive Function
# With degree search i (number of searches):
# Search newly added artists for the current subset of unsearched artists

# **Utility Functions**

In [ ]:
# Clean the collaborations to remove duplicates
for artist in artist_dictionary:
  related_artist = artist_dictionary[artist]['Related Artists']['Artist Collaboration']
  artist_dictionary[artist]['Related Artists']['Artist Collaboration'] = list(set(related_artist))

print(json.dumps(artist_dictionary, indent=4))

In [ ]:
# Store the exising graph into JSON
downloads_path = os.path.expanduser("~")
current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

file_name = os.path.join(downloads_path,f"artist_network_{current_time}.json")

# Writing data to JSON file
with open(file_name, "w") as json_file:
    json.dump(artist_dictionary, json_file, indent=4)

files.download(file_name)
print(f"Data has been written to {file_name}")